In this notebook, we are goin to compute wavelet time-frequency spectrograms for events, baselined to other epochs (e.g. fixation crosses) from the task. We are also going to focus on saving the data out in an efficient matter for subsequent analysis

In [ ]:
# %matplotlib inline
# %reload_ext autoreload
# %autoreload 2

In [ ]:
import numpy as np
import mne
import mne_connectivity
from mne_connectivity import phase_slope_index, seed_target_indices, spectral_connectivity_epochs
from glob import glob
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import zscore, linregress, ttest_ind, ttest_rel, ttest_1samp
import pandas as pd
from mne.preprocessing.bads import _find_outliers
import fooof
# Import plotting functions
from fooof.plts.spectra import plot_spectrum, plot_spectra
# Import the FOOOF object
from fooof import FOOOF
from fooof import FOOOFGroup
import os 
import joblib
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from tqdm import tqdm
from IPython.display import clear_output
import warnings 
from ast import literal_eval

# I only want to see warnings once
warnings.filterwarnings('ignore')

from joblib import delayed, Parallel
from statsmodels.stats import multitest
import re


In [ ]:
# Set some plot defaults for matplotlib (when not using seaborn): 
%matplotlib inline
import  matplotlib.font_manager

path = '/sc/arion/projects/guLab/Salman/Arial.ttf'
matplotlib.font_manager.fontManager.addfont(path)

prop = matplotlib.font_manager.FontProperties(fname=path)

# Set the default font family to the registered custom font
plt.rcParams['font.family'] = prop.get_name()

# Rest of your customization code
plt.rcParams['font.size'] = 20
plt.rcParams['lines.linewidth'] = 1



Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
import sys
sys.path.append('/hpc/users/qasims01/resources/LFPAnalysis')

In [ ]:
from LFPAnalysis import lfp_preprocess_utils, sync_utils, analysis_utils, nlx_utils, oscillation_utils, statistics_utils


## First, get the behavior data

This is from another notebook with a kernel and environment specifically for beahvior

In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site=='UI']
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

## Wavelet TFRs

Here, we will plot some basic TFRs of our behaviorally-locked analysis using a wavelet transform. We can also use these to compute the inter-trial phase coherence (ITC). 

Let's set some parameters first


## Compute for decision-making first

In [ ]:
# Explicitly define a dict of events from your behavioral dataframe

baseline_event = {'baseline_start':[0, 0.75]}
task_events = {'gamble_start':[-1.0, 0],
              'feedback_start': [0, 1.5]}

day = 'Day1'

# Define parameters for TFR: 
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

for subj_id in tqdm(subj_ids):
    lfp_preprocess_utils.compute_and_baseline_tfr(baseline_event, task_events, freqs, n_cycles, 
                             load_path=f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}', 
                             save_path=f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR',
                             IED_artifact_thresh=True, uncaptured_z_thresh=True, output='save')

In [ ]:
event = 'gamble_start'
zpow = mne.time_frequency.read_tfrs(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR/{event}-tfr.h5')[0]




In [ ]:
plot_data = np.nanmean(zpow, axis=0)
fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
times = np.linspace(0, 1.5, plot_data.shape[-1])

im = ax.imshow(plot_data[3, :, :],
              aspect='auto', origin='lower', cmap='Spectral_r', interpolation = 'bicubic')            
fig.colorbar(im, ax=ax)
freq_res = 5
time_res = 250 
ax.set(yticks=np.arange(0, len(freqs))[::freq_res], 
       yticklabels=freqs[::freq_res].astype(int), 
       xticks = np.arange(len(times))[::time_res],
       xticklabels= zpow.times[::time_res])


This is me z-scoring the trialwise , natural log-transformed uV data

In [ ]:
plt.hist(np.nanmean(plot_data[:, np.where((freqs>3) & (freqs<8))[0], :], axis=(1, 2)))
plt.hist(np.nanmean(plot_data[:, np.where((freqs>8) & (freqs<13))[0], :], axis=(1, 2)))
plt.hist(np.nanmean(plot_data[:, np.where((freqs>13) & (freqs<30))[0], :], axis=(1, 2)))
plt.hist(np.nanmean(plot_data[:, np.where((freqs>70) & (freqs<200))[0], :], axis=(1, 2)))

This is me z-scoring the trialwise data

In [ ]:
plt.hist(np.nanmean(plot_data[:, np.where((freqs>3) & (freqs<8))[0], :], axis=(1, 2)))
plt.hist(np.nanmean(plot_data[:, np.where((freqs>8) & (freqs<13))[0], :], axis=(1, 2)))
plt.hist(np.nanmean(plot_data[:, np.where((freqs>13) & (freqs<30))[0], :], axis=(1, 2)))
plt.hist(np.nanmean(plot_data[:, np.where((freqs>70) & (freqs<200))[0], :], axis=(1, 2)))

This is me z-scoring the raw Voltage data against ALL baseline values:

In [ ]:
plt.hist(np.nanmean(plot_data[:, np.where((freqs>3) & (freqs<8))[0], :], axis=(1, 2)))
plt.hist(np.nanmean(plot_data[:, np.where((freqs>8) & (freqs<13))[0], :], axis=(1, 2)))
plt.hist(np.nanmean(plot_data[:, np.where((freqs>13) & (freqs<30))[0], :], axis=(1, 2)))
plt.hist(np.nanmean(plot_data[:, np.where((freqs>70) & (freqs<200))[0], :], axis=(1, 2)))

In [ ]:
plot_data = np.nanmean(zpow, axis=0)
fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
times = np.linspace(0, 1.5, plot_data.shape[-1])

im = ax.imshow(plot_data[0, :, :],
              aspect='auto', origin='lower', cmap='Spectral_r', interpolation = 'bicubic')            
fig.colorbar(im, ax=ax)
freq_res = 5
time_res = 250 
ax.set(yticks=np.arange(0, len(freqs))[::freq_res], 
       yticklabels=freqs[::freq_res].astype(int), 
       xticks = np.arange(len(times))[::time_res],
       xticklabels= zpow.times[::time_res])


In [ ]:
plt.plot(zpow.times, np.nanmean(plot_data[0, 23:, :], axis=0))
plt.plot(zpow.times, np.nanmean(plot_data[0, :10, :], axis=0))

In [ ]:
np.reshape(np.nanmean(plot_data[:, 23:, :], axis=1), -1)

In [ ]:
plt.hist(np.reshape(np.nanmean(plot_data[:, 24:, :], axis=1), -1))

 - power_epochs: a dictionary containing all of your subjects' event TFR, with the buffer cropped out and NORMALIZED with the baseline + event period 

## Do the same for memory epochs

In [ ]:
# Explicitly define a dict of events from your behavioral dataframe

baseline_event = {'baseline_start_mem':[0, 0.75]}
task_events = {'cue_on':[0, 0.5]}

day = 'Day1'

# Define parameters for TFR: 
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

for subj_id in tqdm(subj_ids):
    lfp_preprocess_utils.compute_and_baseline_tfr(baseline_event, task_events, freqs, n_cycles, 
                             load_path=f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}', 
                             save_path=f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR',
                             IED_artifact_thresh=True, uncaptured_z_thresh=True, output='save')

In [ ]:
# Compute baseline_power
for subj_id in subj_ids:
        
    save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
    baseline_epochs_reref = mne.read_epochs(f'{save_path}/{baseline_ev}-epo.fif', preload=True)

    temp_pow  = mne.time_frequency.tfr_morlet(baseline_epochs_reref, freqs=freqs, n_cycles=n_cycles, picks=baseline_epochs_reref.ch_names, 
                                             use_fft=True, n_jobs=-1, output='power', return_itc=False, average=False)
    
    temp_pow.crop(tmin=evs[baseline_ev][0], tmax=evs[baseline_ev][1])
    
    # NAN out the bad data
    # THE following will now LOAD in dataframes that indicate IED and artifact time points in your data
    IED_df = pd.read_csv(f'{save_path}/{baseline_ev}_IED_df.csv') 
    artifact_df = pd.read_csv(f'{save_path}/{baseline_ev}_artifact_df.csv') 

    # Now, let's iterate through each channel, and each ied/artifact, and NaN 100 ms before and after these timepoints
    for ch_ix, ch_name in enumerate(baseline_epochs_reref.ch_names): 
        ied_ev_list = IED_df[ch_name].dropna().index.tolist()
        artifact_ev_list = artifact_df[ch_name].dropna().index.tolist() 
        for ev_ in ied_ev_list: 
            for ied_ in literal_eval(IED_df[ch_name].iloc[ev_]):
                # remove 100 ms before 
                ev_ix_start = np.max([0, np.floor((ied_- 0.1) * baseline_epochs_reref.info['sfreq'])]).astype(int)
                # remove 100 ms after 
                ev_ix_end = np.min([temp_pow.data.shape[-1], np.ceil((ied_ + 0.1) * baseline_epochs_reref.info['sfreq'])]).astype(int)
                temp_pow.data[ev_, ch_ix, :, ev_ix_start:ev_ix_end] = np.nan
        for ev_ in artifact_ev_list: 
            for artifact_ in literal_eval(artifact_df[ch_name].iloc[ev_]):
                # remove 100 ms before 
                ev_ix_start = np.max([0, np.floor((artifact_- 0.1) * baseline_epochs_reref.info['sfreq'])]).astype(int)
                # remove 100 ms after
                ev_ix_end = np.min([temp_pow.data.shape[-1], np.ceil((artifact_ + 0.1) * baseline_epochs_reref.info['sfreq'])]).astype(int)
                temp_pow.data[ev_, ch_ix, :, ev_ix_start:ev_ix_end] = np.nan

    baseline_power_epochs[subj_id][baseline_ev] = temp_pow    

In [ ]:
for subj_id in subj_ids:
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    if not os.path.exists(filepath):
        os.makedirs(filepath)
        
    for event in analysis_evs:
        save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        event_epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif', preload=True)
        
        event_epochs_reref._data = event_epochs_reref._data * 1000000
        
        ch_names = event_epochs_reref.ch_names
    
        temp_pow = mne.time_frequency.tfr_morlet(event_epochs_reref, freqs=freqs, n_cycles=n_cycles, picks=event_epochs_reref.ch_names, 
                                             use_fft=True, n_jobs=-1, output='power', return_itc=False, average=False)
    
        temp_pow.crop(tmin=evs[event][0], tmax=evs[event][1])
        
        # NAN out the bad data
        # THE following will now LOAD in dataframes that indicate IED and artifact time points in your data
        IED_df = pd.read_csv(f'{save_path}/{event}_IED_df.csv') 
        artifact_df = pd.read_csv(f'{save_path}/{event}_artifact_df.csv') 

        # Now, let's iterate through each channel, and each ied/artifact, and NaN 100 ms before and after these timepoints
        for ch_ix, ch_name in enumerate(event_epochs_reref.ch_names): 
            ied_ev_list = IED_df[ch_name].dropna().index.tolist()
            artifact_ev_list = artifact_df[ch_name].dropna().index.tolist() 
            for ev_ in ied_ev_list: 
                for ied_ in literal_eval(IED_df[ch_name].iloc[ev_]):
                    # remove 100 ms before 
                    ev_ix_start = np.max([0, np.floor((ied_- 0.1) * event_epochs_reref.info['sfreq'])]).astype(int)
                    # remove 100 ms after
                    ev_ix_end = np.min([temp_pow.data.shape[-1], np.ceil((ied_ + 0.1) * event_epochs_reref.info['sfreq'])]).astype(int)
                    temp_pow.data[ev_, ch_ix, :, ev_ix_start:ev_ix_end] = np.nan
            for ev_ in artifact_ev_list: 
                for artifact_ in literal_eval(artifact_df[ch_name].iloc[ev_]):
                    # remove 100 ms before 
                    ev_ix_start = np.max([0, np.floor((artifact_- 0.1) * event_epochs_reref.info['sfreq'])]).astype(int)
                    # remove 100 ms after
                    ev_ix_end = np.min([temp_pow.data.shape[-1], np.ceil((artifact_ + 0.1) * event_epochs_reref.info['sfreq'])]).astype(int)
                    temp_pow.data[ev_, ch_ix, :, ev_ix_start:ev_ix_end] = np.nan
        
#         # Let's nan out absurd z-scores (10 std above baseline???) that escaped our artifact detection, noise removal, and baselining
        absurdity_threshold = 10
        max_iter = 10
        large_z_flag=True 
        
        iteration = 0
        while (large_z_flag==True) & (iteration<max_iter): 
            print(f'baseline z-score iteration # {iteration}')
            # Baseline by all the baseline periods in the session
            baseline_corrected_power = lfp_preprocess_utils.baseline_trialwise_TFR(data=temp_pow.data, include_epoch_in_baseline=False,
                                  baseline_mne=baseline_power_epochs[subj_id][baseline_ev].data, 
                                  mode='zscore', ev_axis=0, elec_axis=1, freq_axis=2, time_axis=3)
            
            large_z_mask = np.where(baseline_corrected_power>absurdity_threshold)
            if large_z_mask[0].shape[0] == 0:
                # no more large z
                large_z_flag = False
            else:
                # NaN it out in the event of interest prior to re-running the baseline z-score to prevent
                # contamination of all z's
                temp_pow.data[large_z_mask] = np.nan
                
            iteration +=1
            
        baseline_corrected_power[baseline_corrected_power>absurdity_threshold] = np.nan
        
        zpow = mne.time_frequency.EpochsTFR(event_epochs_reref.info, baseline_corrected_power, 
                                        temp_pow.times, freqs)
        
        zpow.metadata = event_epochs_reref.metadata
        
        
        zpow.save(f'{filepath}/{event}-tfr.h5', overwrite=True)
        

In [ ]:
np.nanmax(baseline_corrected_power)

Let's try to plot individual examples

In [ ]:
analysis_evs = ['feedback_start']
chan_dfs = [] 
glms = []
elec_counter = 0 
for subj_id in subj_ids:
    print(f'Loading subj {subj_id}')
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
    
    
    for event in analysis_evs:
        for chan in elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
            chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')
                
            
            chan_df['unique_elec_id'] = elec_counter
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
#             if event == 'recog_time':
#                 chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
#                 # Make a new response variable, memory outcome 
#                 chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
#                                        'false_alarms']].idxmax(1).to_frame('memory_outcome')                
#                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
#                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
#             else:
            chan_df['zpm'] = (chan_df.DPRIME_x - np.nanmean(chan_df.DPRIME_x)) / (2*np.nanstd(chan_df.DPRIME_x))
            for band in chan_df.fband.unique(): 
                chan_df.loc[chan_df.fband==band, 'ztfr'] = (chan_df.loc[chan_df.fband==band, 'tfr'] - np.nanmean(chan_df.loc[chan_df.fband==band, 'tfr'])) / (2*np.nanstd(chan_df.loc[chan_df.fband==band, 'tfr']))
    
            # try:
            #     chan_df['gamble_rt_log'] = np.log(chan_df['gamble_rt'])
            # except KeyError:
            #     chan_df['gamble_rt_log'] = np.log(chan_df['gamble_rt_gamble'])
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]
            
            elec_counter +=1
            
            chan_dfs.append(chan_df)
        
chan_dfs = pd.concat(chan_dfs)
chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
chan_dfs['hit_binom'] = chan_dfs['hits'].astype(int)
chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5

## Time-average the TFR data: 

8/28/2024: Let's split the feedback data into 500 ms blocks.

In [ ]:
progress_bar = tqdm(subj_ids, ascii=True, desc='Saving TFR dataframes')
analysis_evs = ['cue_on'] 

# power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids}
# blocks = [[0, 1.5], ]
blocks = [[0, 0.5], ]

for subj_id in subj_ids:
    for day in ['Day1']:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
        for event in analysis_evs:

            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            if not os.path.exists(f'{filepath}/{event}/plots'):
                os.makedirs(f'{filepath}/{event}/plots')
            if not os.path.exists(f'{filepath}/{event}/dfs'):
                os.makedirs(f'{filepath}/{event}/dfs')

            # power_epochs = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
            power_epochs = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
            
            # 5/3/2024
            # If we care about both RPE and memory in encoding (we dont yet!)
            if event == 'cue_on':
                metadata = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].sort_values(by='trials_mem')
            else: 
                metadata = learn_df[(learn_df.participant==subj_id)]
                # epochs_to_analyze = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble']).trials_gamble.values - 1
                # power_epochs = power_epochs[epochs_to_analyze.astype(int)]

            power_epochs.metadata = metadata

            for chan in power_epochs.ch_names:
                pow_df_whole = power_epochs.copy().pick([chan]).to_data_frame()
                
                for block in blocks:

                    # If i want to truncate: 
                    pow_df = pow_df_whole[(pow_df_whole.time>block[0]) & (pow_df_whole.time<=block[1])]

                    pow_df['fband'] = pow_df.freq.apply(lambda x: 'low_theta' if x<4 else 'high_theta' if (x>=4) & (x<8) else 'alpha' if (x>=8) & (x<13) else 'beta' if (x>=13) & (x<30) else 'gamma' if (x>=30) & (x<70) else 'hfa')

                    pow_df = pow_df.groupby(['fband','epoch']).mean().reset_index()

                    tfr_df = pow_df.drop(columns=['time', 'freq']).rename(columns={'epoch':'trial', f'{chan}':'tfr'})

                    power_epochs.metadata['trial'] = tfr_df['trial'].unique()
                    tfr_df = tfr_df.merge(power_epochs.metadata, on=['trial'])

                    tfr_df.insert(0,'channel', chan)
                    tfr_df.insert(0,'region', elec_df[elec_df.label==chan].salman_region.values[0])
    #                 tfr_df.insert(0,'participant', subj_id)
                    tfr_df['event'] = event

                    tfr_df.to_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_time{block[0]}-{block[1]}.csv', index=False)


In [ ]:
print('true')

## Example 1: HPC electrodes contrasting negative and positive RPE (HFA) during encoding

In [ ]:
# Load the relevant bayesian regression results and sort the electrodes so you can identify plotting candidates: 
load_dir = '/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
region = 'HPC'
band = 'hfa'
task_phase = 'encoding'

summary_df = pd.read_csv(f'{load_dir}/{region}_{band}_{task_phase}_fully_hierarchical_t_summary.csv')
summary_df.rename(columns={'Unnamed: 0':'parameter'}, inplace=True)

In [ ]:
summary = summary_df[summary_df['parameter'].str.contains(r'\bzrpe\|unique_elec_id\[')]


In [ ]:
# Sort the electrodes from LOW to HIGH individual beta values 
sorted_elecs = [int(re.search(r'\[(\d+)\]', x).group(1)) if re.search(r'\[(\d+)\]', x) else None for x in summary.sort_values(by=['mean'])['parameter']]


In [ ]:
sorted_elecs[-1] # is the strongest effect 

In [ ]:
# which electrode is this? 
subj_id = chan_dfs[chan_dfs.unique_elec_id==sorted_elecs[-1]].participant.unique()[0]
chan_id = chan_dfs[chan_dfs.unique_elec_id==sorted_elecs[-1]].channel.unique()[0]

event = 'feedback_start'

In [ ]:
subj_id

In [ ]:
chan_id

In [ ]:
filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'

power_epochs = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]

In [ ]:
ch_ix = power_epochs.ch_names.index(chan_id)
freqs = np.logspace(*np.log10([2, 200]), num=30)
freq_ix = np.where(freqs[freqs<9])[0]

In [ ]:
# get the behavior: 
subj_df = chan_dfs[(chan_dfs.unique_elec_id==sorted_elecs[-1]) & (chan_dfs.fband==band)]


In [ ]:
subj_df.trial.values

In [ ]:
power_epochs.data[subj_df.trial.values, ch_ix, freq_ix, :].shape

## EXample 2: OFC electrodes contrasting RPE AND Memory (theta) during encoding

In [ ]:
band = 'hfa'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')



In [ ]:
roi = 'OFC'

In [ ]:
exemplar_df = gamble_model_df[gamble_model_df.region==f'{roi}'].sort_values(by='phit:zrpe', ascending=False).iloc[0:10]

In [ ]:
exemplar_df

In [ ]:
# plot the 5 best electrodes: 
from scipy.interpolate import griddata


analysis_evs = 'feedback_start'

# cond_name = 'rpe'

# conditions = ['(rpe>0)',
#              '(rpe<0)']

cond_name = 'SME_posRPE'

conditions = ['(rpe>0) & (hits==1)',
             '(rpe>0) & (hits==0)']

freqs = np.logspace(*np.log10([2, 200]), num=30)


for row in exemplar_df.itertuples(): 
    
    fig, ax = plt.subplots(1, 2, figsize=(20, 6), dpi=300)
        
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{row.participant}/scratch/TFR'
    power_epoch = mne.time_frequency.read_tfrs(f'{filepath}/{analysis_evs}-tfr.h5')[0]
    
    
    if analysis_evs == 'feedback_start': 
        if cond_name == 'rpe':
            power_epoch.metadata = learn_df[(learn_df.participant==row.participant)]
        else:
            epochs_to_analyze = combined_df[(combined_df.participant==row.participant) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_gamble').reset_index(drop=True).trials_gamble.values - 1

            power_epoch = power_epoch[epochs_to_analyze.astype(int)]

            power_epoch.metadata = combined_df[(combined_df.participant==row.participant) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_gamble').reset_index(drop=True)
            
    
    times = np.linspace(0, 1.5, power_epoch.data.shape[-1])
    
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{row.participant}/Day1_reref_elec_df')
    
    for ix, cond in enumerate(conditions[::-1]):
        plot_data = np.nanmean(power_epoch.copy()[cond].pick(f'{row.channel}').data, axis=(1))
        
        for trial in range(plot_data.shape[0]):
            # smooth over NaN datapoints for visualization
            
            # Get the coordinates of valid and missing data points
            x, y = np.indices(plot_data[trial].shape)
            valid_mask = ~np.isnan(plot_data[trial])

            # Interpolate NaNs based on surrounding valid data points
            interpolated_array = griddata(
                (x[valid_mask], y[valid_mask]),  # Coordinates of known values
                plot_data[trial][valid_mask],               # Values at known coordinates
                (x, y),                          # All coordinates (grid to interpolate over)
                method='linear'                  # Choose method: 'linear', 'nearest', 'cubic'
            )
            
            plot_data[trial] = interpolated_array
    
        plot_data = np.nanmean(plot_data, axis=0) 
        
        im = ax[ix].imshow(plot_data,
                aspect='auto', origin='lower', cmap='Spectral_r', interpolation = 'gaussian',
                           vmin=-0.75, vmax=0.75)            
        ax[ix].set(xlabel='Time (s)', ylabel='Frequency', title=f'{cond}')

        freq_res = 5
        time_res = 250 
        ax[ix].set(yticks=np.arange(0, len(freqs))[::freq_res], 
               yticklabels=freqs[::freq_res].astype(int), 
               xticks = np.arange(len(times))[::time_res],
               xticklabels= times[::time_res])

        ax[ix].hlines(np.where(np.round(freqs)==8)[0][0], 0, times[-1])
        ax[ix].hlines(np.where(np.round(freqs)==66)[0][0], 0, times[-1])
        fig.colorbar(im, ax=ax[ix])
        
    plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{analysis_evs}_TFR_{cond_name}_{roi}_{row.participant}_{row.channel}.pdf', dpi=300)

        
    

In [ ]:
combined_df[(combined_df.participant==row.participant) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_gamble').reset_index(drop=True)

In [ ]:
# # Sort the electrodes from LOW to HIGH individual beta values 
# sorted_elecs = [int(re.search(r", (\d+)\]", x).group(1)) if re.search(r", (\d+)\]", x) else None for x in summary.sort_values(by=['mean'])['parameter']]


I have now computed the TFR computations for both decision-making and memory

In [ ]:
# rois = ['dmPFC', 'OFC', 'HPC', 'AMY', 'ACC', 'dlPFC']
rois = ['OFC']

In [ ]:
# Let's load the power epochs for specific conditions: 

analysis_evs = ['feedback_start']

# band definitions for y-axis
# yticks = [4, 8, 13, 30, 60, 120]

# task condition to contrast 

cond_name = 'rpe'

conditions = ['(rpe>0)',
             '(rpe<0)']

# cond_name = 'SME_negRPE'

# conditions = ['(rpe<0) & (hits==1)',
#              '(rpe<0) & (hits==0)']



tfr_group_data = {f'{a}': {f'{b}': [] for b in conditions} for a in rois}

power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids}

channel_names = [] 
subj_names = [] 

for subj_id in subj_ids: 
    for event in analysis_evs:
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        power_epochs[subj_id][event] = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
                
        # replace IED metadata with behavioral metadata
        if ('SME' in cond_name) or (event in ['cue_on', 'recog_time']):
            epochs_to_analyze = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_mem').reset_index(drop=True).trials_mem.values - 1

            power_epochs[subj_id][event] = power_epochs[subj_id][event][epochs_to_analyze.astype(int)]

            power_epochs[subj_id][event].metadata = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_mem').reset_index(drop=True)
        else:
            power_epochs[subj_id][event].metadata = learn_df[(learn_df.participant==subj_id)]
            
        # Get electrode df 
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
        
        
        progress_bar = tqdm(elec_df.label, ascii=True, desc='Aggregating TFRs')

        for elec_ix, chan in enumerate(progress_bar):
            region = elec_df.salman_region.iloc[elec_ix]
            if region in rois:
                for ix, cond in enumerate(conditions):
                    # Set the times for 
                    times = power_epochs[subj_id][event].times
                    plot_data = np.nanmean(np.nanmean(power_epochs[subj_id][event][cond].pick([chan]).data, axis=0), axis=0)
                    tfr_group_data[region][cond].append(plot_data)    
                    
                    channel_names.append(chan)
                    subj_names.append(subj_id)
                    

In [ ]:
freqs = np.logspace(*np.log10([2, 200]), num=30)

for region in rois: 
    fig, ax = plt.subplots(1, 2, figsize=(20, 6), dpi=300)
    for ix, cond in enumerate(conditions[::-1]): 
        # Set the times for 
        n_elecs = np.squeeze(np.stack(tfr_group_data[region][cond])).shape[0]
        times = np.linspace(0, 1.5, np.squeeze(np.stack(tfr_group_data[region][cond])).shape[2])

#         if cond_name == 'SME':
#             plot_data = np.nanmedian(np.nanmedian(np.squeeze(np.stack(tfr_group_data[region][cond])), axis=0), axis=0)
#         else:
        dat_ = np.squeeze(np.stack(tfr_group_data[region][cond]))
#         dat_[dat_>3] = np.nan
        plot_data = np.nanmean(dat_, axis=0)

#         if region in ['HPC', 'ACC']:
#             im = ax[ix].imshow(plot_data,
#                     aspect='auto', origin='lower', cmap='inferno', interpolation = 'bicubic', vmin =-0.12, vmax=0.12)  
#         elif region in ['AMY']:
#             im = ax[ix].imshow(plot_data,
#                     aspect='auto', origin='lower', cmap='inferno', interpolation = 'bicubic', vmin =-0.1, vmax=0.1)            
#         elif region in ['dmPFC']:
#             im = ax[ix].imshow(plot_data,
#                     aspect='auto', origin='lower', cmap='inferno', interpolation = 'bicubic', vmin =-0.17, vmax=0.17)         
#         else:

        if region in ['OFC']:
            vmin = -0.075
            vmax = 0.125
        else:
            vmin = -0.1
            vmax = 0.1

        im = ax[ix].imshow(plot_data,
                aspect='auto', origin='lower', cmap='Spectral_r', interpolation = 'bicubic', vmin = vmin, vmax= vmax)            
        ax[ix].set(xlabel='Time (s)', ylabel='Frequency', title=f'{cond}_{region}_nelec={n_elecs}')
        
        # Set y-axis to log scale
#         ax[ix].set_yscale('log')

        # Set custom y-axis ticks and labels
#         yticks = [2, 10, 30, 50, 200]  # Specify the desired y-axis ticks
#         yticklabels = ['2', '10', '30', '50', '200'] # Specify the labels for the ticks
        freq_res = 5
        time_res = 250 
        ax[ix].set(yticks=np.arange(0, len(freqs))[::freq_res], 
               yticklabels=freqs[::freq_res].astype(int), 
               xticks = np.arange(len(times))[::time_res],
               xticklabels= times[::time_res])

        ax[ix].hlines(np.where(np.round(freqs)==8)[0][0], 0, times[-1])
        ax[ix].hlines(np.where(np.round(freqs)==66)[0][0], 0, times[-1])
        fig.colorbar(im, ax=ax[ix])
        plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{event}_TFR_{cond_name}_{region}.pdf', dpi=300)

        
        

In [ ]:
plot_data.shape

In [ ]:
freqs[2]

In [ ]:
np.nanmin([np.nanmin(tfr_group_data[region][cond]) for cond in conditions])

If computing the SME, don't mean before the statistical contrast.... but do so before visualizing the TFR...? 

In [ ]:
freqs = np.logspace(*np.log10([2, 200]), num=30)

for region in rois: 
    with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/feedback_TFR_{cond_name}_bychan.pdf') as pdf:
        for ch_ in range((len(channel_names)//2) - 1):
            fig, ax = plt.subplots(1, 2, figsize=(20, 6), dpi=300)
            for ix, cond in enumerate(conditions): 
                # Set the times for 
                n_elecs = np.squeeze(np.stack(tfr_group_data[region][cond])).shape[0]
                times = np.linspace(0, 1.5, np.squeeze(np.stack(tfr_group_data[region][cond])).shape[2])
                dat_ = np.squeeze(np.stack(tfr_group_data[region][cond]))
                im = ax[ix].imshow(dat_[ch_, :, :],
                          aspect='auto', origin='lower', cmap='inferno', interpolation = 'bicubic')            
                ax[ix].set(xlabel='Time (s)', ylabel='Frequency', title=f'{cond}_{region}_nelec={n_elecs}')
                freq_res = 5
                time_res = 250 
                ax[ix].set(yticks=np.arange(0, len(freqs))[::freq_res], 
                       yticklabels=freqs[::freq_res].astype(int), 
                       xticks = np.arange(len(times))[::time_res],
                       xticklabels= times[::time_res])
            fig.colorbar(im, ax=ax[ix])
            pdf.savefig(fig)
            plt.close()

        
        

In [ ]:
rois = ['HPC', 'dmPFC', 'ACC', 'OFC', 'AMY']

In [ ]:
for region in rois: 
    # if region == 'TC':
    #     region = (('Temporal') | ('Temporal Pole') | ('STS'))
    fig, ax = plt.subplots(1, 1, figsize=(6, 3), dpi=300, sharey=True, sharex=True)
    for ix, cond in enumerate(conditions): 
        # Set the times for 
        n_elecs = np.squeeze(np.stack(tfr_group_data[region][cond])).shape[0]
        times = np.linspace(0, 0.5, np.squeeze(np.stack(tfr_group_data[region][cond])).shape[2])

#         if cond_name == 'SME':
#             plot_data = np.nanmedian(np.nanmedian(np.squeeze(np.stack(tfr_group_data[region][cond])), axis=0), axis=0)
#         else:
#         plot_data = np.nanmean(np.squeeze(np.stack(tfr_group_data[region][cond])), axis=0)

        mean_hfa = np.nanmean(np.squeeze(np.stack(tfr_group_data[region][cond]))[:, (freqs>70) & (freqs<=200), :], axis=(0,1))
        
        se_hfa = np.nanstd(np.squeeze(np.stack(tfr_group_data[region][cond]))[:, (freqs>70) & (freqs<=200), :], axis=(0,1)) / (np.sqrt(n_elecs*np.sum((freqs>70) & (freqs<=200))))
        
#         smoothed_hfa = gaussian_filter1d(mean_hfa, sigma=1)
        
        ax.plot(times, mean_hfa)
        plt.fill_between(times, mean_hfa-se_hfa, mean_hfa+se_hfa, alpha = 0.2)

#         ax.errorbar(times, mean_hfa, yerr=se_hfa)
        ax.set(xlabel='Time (s)', ylabel='Power (z)', title=f'{region}_nelec={n_elecs}')
        # fig.colorbar(im, ax=ax[ix])
        sns.despine()
        plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{event}_hfa_{cond_name}_{region}.pdf', dpi=300)


In [ ]:
amy_data1 = np.nanmean(np.squeeze(np.stack(tfr_group_data['AMY']['(rpe>0)']))[:, (freqs>70) & (freqs<=200), :], axis=1)


In [ ]:
amy_data2 = np.nanmean(np.squeeze(np.stack(tfr_group_data['AMY']['(rpe<0)']))[:, (freqs>70) & (freqs<=200), :], axis=1)


In [ ]:
from neurodsp.spectral import compute_spectrum
from neurodsp.plts.spectral import plot_power_spectra


In [ ]:
freq_mean, psd_mean1 = compute_spectrum(amy_data1, 500, method='welch', avg_type='mean', nperseg=500*2)



In [ ]:
freq_mean, psd_mean2 = compute_spectrum(amy_data2, 500, method='welch', avg_type='mean', nperseg=500*2)


In [ ]:
plt.plot(freq_mean, psd_mean1.mean(axis=0))
plt.plot(freq_mean, psd_mean2.mean(axis=0))

In [ ]:
np.nanmean(np.squeeze(np.stack(tfr_group_data['AMY'][('(rpe<0)')]))[:, (freqs>70) & (freqs<=200), :])

In [ ]:
for region in rois: 
    # if region == 'TC':
    #     region = (('Temporal') | ('Temporal Pole') | ('STS'))
    fig, ax = plt.subplots(1, 1, figsize=(6, 3), dpi=300, sharey=True, sharex=True)
    for ix, cond in enumerate(conditions): 
        # Set the times for 
        n_elecs = np.squeeze(np.stack(tfr_group_data[region][cond])).shape[0]
        times = np.linspace(0, 1.5, np.squeeze(np.stack(tfr_group_data[region][cond])).shape[2])

#         if cond_name == 'SME':
#             plot_data = np.nanmedian(np.nanmedian(np.squeeze(np.stack(tfr_group_data[region][cond])), axis=0), axis=0)
#         else:
#         plot_data = np.nanmean(np.squeeze(np.stack(tfr_group_data[region][cond])), axis=0)

        mean_hfa = np.nanmean(np.squeeze(np.stack(tfr_group_data[region][cond]))[:, (freqs>13) & (freqs<=30), :], axis=(0,1))
        
        se_hfa = np.nanstd(np.squeeze(np.stack(tfr_group_data[region][cond]))[:, (freqs>13) & (freqs<=30), :], axis=(0,1)) / (np.sqrt(n_elecs*np.sum((freqs>70) & (freqs<=200))))
        
        smoothed_hfa = gaussian_filter1d(mean_hfa, sigma=1)
        
        ax.plot(times, mean_hfa)
        plt.fill_between(times, mean_hfa-se_hfa, mean_hfa+se_hfa, alpha = 0.2)

#         ax.errorbar(times, mean_hfa, yerr=se_hfa)
        ax.set(xlabel='Time (s)', ylabel='Power (z)', title=f'{region}_nelec={n_elecs}')
        # fig.colorbar(im, ax=ax[ix])
        sns.despine()
        plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/feedback_beta_{cond_name}_{region}.pdf', dpi=300)


In [ ]:
for region in rois: 
    # if region == 'TC':
    #     region = (('Temporal') | ('Temporal Pole') | ('STS'))
    fig, ax = plt.subplots(1, 1, figsize=(6, 3), dpi=300, sharey=True, sharex=True)
    for ix, cond in enumerate(conditions): 
        # Set the times for 
        n_elecs = np.squeeze(np.stack(tfr_group_data[region][cond])).shape[0]
        times = np.linspace(0, 1.5, np.squeeze(np.stack(tfr_group_data[region][cond])).shape[2])

#         if cond_name == 'SME':
#             plot_data = np.nanmedian(np.nanmedian(np.squeeze(np.stack(tfr_group_data[region][cond])), axis=0), axis=0)
#         else:
#         plot_data = np.nanmean(np.squeeze(np.stack(tfr_group_data[region][cond])), axis=0)

        mean_hfa = np.nanmean(np.squeeze(np.stack(tfr_group_data[region][cond]))[:, (freqs>3) & (freqs<=8), :], axis=(0,1))
        
        se_hfa = np.nanstd(np.squeeze(np.stack(tfr_group_data[region][cond]))[:, (freqs>3) & (freqs<=8), :], axis=(0,1)) / (np.sqrt(n_elecs*np.sum((freqs>70) & (freqs<=200))))
        
        smoothed_hfa = gaussian_filter1d(mean_hfa, sigma=1)
        
        ax.plot(times, mean_hfa)
        plt.fill_between(times, mean_hfa-se_hfa, mean_hfa+se_hfa, alpha = 0.2)

#         ax.errorbar(times, mean_hfa, yerr=se_hfa)
        ax.set(xlabel='Time (s)', ylabel='Power (z)', title=f'{region}_nelec={n_elecs}')
        # fig.colorbar(im, ax=ax[ix])
        sns.despine()
        plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/feedback_theta_{cond_name}_{region}.pdf', dpi=300)


In [ ]:
np.nanmean(np.squeeze(np.stack(tfr_group_data[region][cond]))[:, (freqs>70) & (freqs<=200), :], axis=(0,1))

In [ ]:
np.squeeze(np.stack(tfr_group_data[region][cond])).shape

In [ ]:
from scipy.ndimage import gaussian_filter1d

Plot individual electrodes: 

In [ ]:
subj_id = 'MS036'
ch_name = 'rhplt2-rhplt3'
event = 'feedback_start'
filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR' 
power_epochs = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
power_epochs.metadata = learn_df[(learn_df.participant==subj_id)]

In [ ]:
ch_data = np.squeeze(power_epochs.copy().pick(ch_name)._data)

In [ ]:
plot_data = np.nanmean(ch_data[:, freqs>70
                               , :], axis=1)

In [ ]:
plt.plot(np.nanmean(plot_data[power_epochs.metadata["rpe"]>0], axis=0), 'k')
plt.plot(np.nanmean(plot_data[power_epochs.metadata["rpe"]<0], axis=0), 'r')

In [ ]:
sns.regplot(x=power_epochs.metadata["rpe"], y=np.nanmean(plot_data, axis=1))

In [ ]:
sort_order = np.argsort(power_epochs.metadata["rpe"])
power_epochs.metadata["rpe"][sort_order]

In [ ]:
# sort by RPE
plot_data = plot_data[sort_order.values, :]

In [ ]:
plot_data[~np.isnan(plot_data).any(axis=1)].max()

In [ ]:
im = plt.imshow(plot_data[~np.isnan(plot_data).any(axis=1)],
                      extent=[times[0], times[-1], 0 , plot_data[~np.isnan(plot_data).any(axis=1)].shape[0]],
                      aspect='auto', origin='lower', cmap='RdBu_r', vmin = -2, vmax=2, interpolation='bicubic')
fig.colorbar(im)

Conduct the two-sample permutation test. This is most handy if your categories are naturally discrete, or helpful to bin. It is less good if you woud actually get statistical value from continuous stats.

In [ ]:
# freqs = np.logspace(*np.log10([4, 120]), num=30)

# conditions = ['(rpe>0)',
#              '(rpe<0)']
# conditions = ['(rpe>0) & (hits==1)',
#              '(rpe>0) & (hits==0)']

cond_name = 'SME'
        
# times = np.linspace(0, 1.5, np.stack(tfr_group_data[region][cond]).shape[2])
    
for region in rois: 
    # # Average the data in each condition across channels 
    if 'SME' in cond_name:
        X = [np.squeeze(np.vstack(tfr_group_data[region][conditions[0]])),
             np.squeeze(np.vstack(tfr_group_data[region][conditions[1]]))]
    else:
        X = [np.stack(tfr_group_data[region][conditions[0]]), 
             np.stack(tfr_group_data[region][conditions[1]])]
        
    F_obs, clusters, cluster_p_values, H0 = \
mne.stats.permutation_cluster_test(X, n_permutations=1000, out_type='mask', verbose=True,
                                   threshold=6,
                                   seed=1, n_jobs=1, tail=0)
    
    # Create new stats image with only significant clusters
    fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=300)
    
    # Average the data in each condition across epochs for plotting
    evoked_power_1 = np.nanmean(X[0], axis=0)
    evoked_power_2 = np.nanmean(X[1], axis=0)
    evoked_power_contrast = evoked_power_1 - evoked_power_2
    signs = np.sign(evoked_power_contrast)

    F_obs_plot = np.nan * np.ones_like(F_obs)
    for c, p_val in zip(clusters, cluster_p_values):
        if p_val <= 0.05:
            F_obs_plot[c] = F_obs[c] * signs[c]

    ax.imshow(F_obs,
              extent=[times[0], times[-1], freqs[0], freqs[-1]],
              aspect='auto', origin='lower', cmap='Spectral_r', alpha =0.55)
    max_F = np.nanmax(abs(F_obs_plot))
    c= ax.imshow(F_obs_plot,
              extent=[times[0], times[-1], freqs[0], freqs[-1]],
              aspect='auto', origin='lower', cmap='Spectral_r',
              vmin=-max_F, vmax=max_F)

    ax.set(yticks=yticks, xlabel='Time (s)', ylabel='Frequency', title=f'{region}_{cond_name}')
    fig.colorbar(c, ax=ax)
    plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/feedback_clustperm_{cond_name}_{region}.pdf', dpi=300)


## Memory TFRs contrast RPEs

In [ ]:
# # Let's load the power epochs: 

# analysis_evs = ['recog_time']

# # band definitions for y-axis
# yticks = [4, 8, 13, 30, 60, 120]

# # task condition to contrast 
# conditions = ['(rpe<0)&(hits==True)',
#              '(rpe<0)&(hits==False)']

# cond_name = 'nrpe_memory'

# tfr_group_data = {f'{a}': {f'{b}': [] for b in conditions} for a in rois}

# power_epochs = {f'{a}': {f'{b}': {f'{c}': np.nan for c in days} for b in analysis_evs} for a in subj_ids}

# for subj_ix, subj_id in enumerate(subj_ids):
#     for day in ['Day1']: 
#         if (day=='Day2') & (subj_days[subj_ix]==0):
#             continue
#         else:
#             for event in analysis_evs:
#                 filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
#                 power_epochs[subj_id][event][day] = mne.time_frequency.read_tfrs(f'{filepath}/{event}_{day}-tfr.h5')[0]

#                 power_epochs[subj_id][event][day].metadata = combined_df[(combined_df.participant==subj_id) & (combined_df.condition==day)]

#                 # Get electrode df 
#                 elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')

#                 progress_bar = tqdm(elec_df.label, ascii=True, desc='Aggregating TFRs')

#                 for elec_ix, chan in enumerate(progress_bar):
#                     region = elec_df.salman_region.iloc[elec_ix]
#                     if region in rois:
#                         for ix, cond in enumerate(conditions):
#                             # Set the times for 
#                             times = power_epochs[subj_id][event][day].times
#                             plot_data = power_epochs[subj_id][event][day][cond].pick_channels([chan]).data
#                             tfr_group_data[region][cond].append(plot_data)                    


In [ ]:
# freqs = np.logspace(*np.log10([4, 120]), num=30)

# for region in rois: 
#     fig, ax = plt.subplots(1, 2, figsize=(20, 6), dpi=300)
#     for ix, cond in enumerate(conditions): 
#         # Set the times for 
#         n_elecs = np.squeeze(np.vstack(tfr_group_data[region][cond])).shape[0]
#         # times = np.linspace(0, 1.5, np.squeeze(np.vstack(tfr_group_data[region][cond])).shape[2])

#         plot_data = np.nanmean(np.squeeze(np.vstack(tfr_group_data[region][cond])), axis=0)
#         im = ax[ix].imshow(plot_data,
#                   extent=[times[0], times[-1], freqs[0], freqs[-1]],
#                   aspect='auto', origin='lower', cmap='Spectral_r', 
#                            vmin =-0.1, vmax =0.1, interpolation = 'bicubic')
#         ax[ix].set(yticks=yticks, xlabel='Time (s)', 
#                    ylabel='Frequency', title=f'{cond}_{region}_nelec={n_elecs}')
#         fig.colorbar(im, ax=ax[ix])
#         plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/recog_time_TFR_{cond_name}_{region}.pdf', 
#                     dpi=300)

        
        

In [ ]:
# freqs = np.logspace(*np.log10([4, 120]), num=30)

# cond_name = 'nrpe_memory'
        
# # times = np.linspace(0, 1.5, np.stack(tfr_group_data[region][cond]).shape[2])
    
# for region in rois: 
#     # # Average the data in each condition across channels 
#     if 'memory' in cond_name:
#         X = [np.squeeze(np.vstack(tfr_group_data[region][conditions[0]])),
#              np.squeeze(np.vstack(tfr_group_data[region][conditions[1]]))]
#     else:
#         X = [np.stack(tfr_group_data[region][conditions[0]]), 
#              np.stack(tfr_group_data[region][conditions[1]])]
        
#     F_obs, clusters, cluster_p_values, H0 = \
# mne.stats.permutation_cluster_test(X, n_permutations=500, out_type='mask', verbose=True,
#                                    threshold=6,
#                                    seed=1, n_jobs=1, tail=0)
    
#     # Create new stats image with only significant clusters
#     fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=300)
    
#     # Average the data in each condition across epochs for plotting
#     evoked_power_1 = np.nanmean(X[0], axis=0)
#     evoked_power_2 = np.nanmean(X[1], axis=0)
#     evoked_power_contrast = evoked_power_1 - evoked_power_2
#     signs = np.sign(evoked_power_contrast)

#     F_obs_plot = np.nan * np.ones_like(F_obs)
#     for c, p_val in zip(clusters, cluster_p_values):
#         if p_val <= 0.05:
#             F_obs_plot[c] = F_obs[c] * signs[c]

#     ax.imshow(F_obs,
#               extent=[times[0], times[-1], freqs[0], freqs[-1]],
#               aspect='auto', origin='lower', cmap='Spectral_r', alpha =0.55)
#     max_F = np.nanmax(abs(F_obs_plot))
#     c= ax.imshow(F_obs_plot,
#               extent=[times[0], times[-1], freqs[0], freqs[-1]],
#               aspect='auto', origin='lower', cmap='Spectral_r',
#               vmin=-max_F, vmax=max_F)

#     ax.set(yticks=yticks, xlabel='Time (s)', ylabel='Frequency', title=f'{region}_{cond_name}')
#     fig.colorbar(c, ax=ax)
#     plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/recog_time_clustperm_{cond_name}_{region}.pdf', dpi=300)


## Statistical Analyses of TFRs:

At this stage, you should **heavily** brainstorm the statistics you want to do, before you start writing any code. What is the goal of your analysis? What would actually allow you to show what you want to show?

## One-sample cluster permutation test


A one sample cluster permutation test measures whether a spectrogram differs significantly from a particular mean (typically 0). 

In [ ]:
subj_id = 'MS012'
# region = 'frontal'

# save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
# # Get electrode df 
# elec_path = f'{base_dir}/projects/guLab/Salman/EMU/{subj_id}/anat/'
# # electrode files could either be csv or excel
# elec_files = glob(f'{elec_path}/*.csv') + glob(f'{elec_path}/*.xlsx')
# # There should really only be one 
# elec_file = elec_files[0]
# elec_data = lfp_preprocess_utils.load_elec(elec_file)

# anode_list = [x.split('-')[0] for x in epochs_all_subjs_all_evs[subj_id][event].ch_names]
# elec_df = elec_data[elec_data.label.str.lower().isin(anode_list)]
# # Shawn noticed the following would incorrectl y label electrodes if they're not in the same order as .ch_names! 
# # elec_df['label'] = epochs_all_subjs_all_evs[subj_id][event].ch_names
# # Fix: 
# elec_df['label'] = elec_df['label'].apply(lambda x: next((string for string in epochs_all_subjs_all_evs[subj_id][event].ch_names if x in string), np.nan))
    
# picks = analysis_utils.select_picks_rois(elec_df, region)

for event in analysis_evs:
    
    # Average the data in each condition across channels 
    X = np.nanmean(power_epochs[subj_id][event].copy().pick_channels([chan]).data, axis=1)
    
    F_obs, clusters, cluster_p_values, H0 = \
    mne.stats.permutation_cluster_1samp_test(X, n_permutations=500, out_type='mask', verbose=True)
    
    if any(cluster_p_values<=0.05):
#     print(region)
        # Create new stats image with only significant clusters
        fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=300)

        times =  power_epochs[subj_id][event].times


        # Average the data in each condition across epochs for plotting
        evoked_power = np.nanmean(X, axis=0)
    #     evoked_power_2 = np.nanmean(X[1], axis=0)
    #     evoked_power_contrast = evoked_power_1 - evoked_power_2
        signs = np.sign(evoked_power)

        F_obs_plot = np.nan * np.ones_like(F_obs)
        for c, p_val in zip(clusters, cluster_p_values):
            if p_val <= 0.05:
                F_obs_plot[c] = F_obs[c] * signs[c]

        ax.imshow(F_obs,
                  extent=[times[0], times[-1], freqs[0], freqs[-1]], interpolation = 'Bicubic',
                  aspect='auto', origin='lower', cmap='gray')
        max_F = np.nanmax(abs(F_obs_plot))
        ax.imshow(F_obs_plot,
                  extent=[times[0], times[-1], freqs[0], freqs[-1]],
                  aspect='auto', origin='lower', cmap='RdBu_r',
                  vmin=-max_F, vmax=max_F)

        ax.set(yticks=yticks, xlabel='Time (s)', ylabel='Frequency', title=f'{subj_id}_{chan}_{event}')

            # # ax.set_title(f'Induced power ({ch_name})')
    
    

In [ ]:
from scipy.stats import ttest_ind
def stat_fun(a, b):
      return ttest_ind(a, b, nan_policy='omit')[0]


In [ ]:
sig_cells = []
filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/MS012/scratch/TFR'
event = 'gamble_start'
power_epochs = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]


from joblib import Parallel, delayed

# Assume 'data' is your array of shape (80, 30, 750)

# Parallel processing for efficiency
results = Parallel(n_jobs=-1)(
    delayed(perform_regression)(i, j)
    for i in range(750) for j in range(30)
)


for ch in range(power_epochs.data.shape[1]):
    X = [np.squeeze(power_epochs[learn_df[learn_df.participant=='MS012'].gamble==1].data[:, ch, :, :]),
                                       np.squeeze(power_epochs[learn_df[learn_df.participant=='MS012'].gamble==0].data[:, ch, :, :])]


    t_obs, clusters, cluster_p_values, h0 = mne.stats.permutation_cluster_test(X,n_permutations=500, out_type='mask', verbose=True,
                                                                               seed=1, n_jobs=-1, tail=0, stat_fun = stat_fun)
    if len(cluster_p_values) > 0:
        if any(cluster_p_values <= 0.05):
            sig_cells.append(power_epochs.ch_names[ch])

In [ ]:
print(sig_cells)